# Estudio de segmentación de tejidos en DFUTissue (Colab / GPU)

Corre en Colab con **GPU T4** (`Entorno de ejecución -> Cambiar tipo -> T4 GPU`).
Tres experimentos; todo se guarda directo a Google Drive.

1. **Eficiencia de anotación (T3+T4, bien hecho).** ¿Cuántas máscaras densas hacen falta,
   *aporta algo la anotación débil*, y *conviene filtrarla*? Cuatro curvas Dice-vs-N,
   comparadas en la misma corrida:
   - `solo_supervisado` — solo N máscaras, nada más (control: el modelo aprendiendo de N muestras).
   - `mixto` — N máscaras + **todas** las (78−N) etiquetas débiles.
   - `mixto ≤1` — N máscaras + etiquetas débiles **solo de imágenes con ≤1 tejido** (idea de Adrián:
     una etiqueta `[1,1,1]` no da señal discriminativa a la pérdida LSE-pool+BCE; una `[1,0,0]` sí).
   - `mixto ≤2` — igual con ≤2 tejidos.

   Brecha `mixto` − `solo_supervisado` = valor real de la débil. Brecha `mixto` − `mixto ≤k` = si conviene filtrar.
   *Aviso:* en DFUTissue solo ~12 imágenes de train tienen 1 tejido y ~38 tienen ≤2, así que la curva
   `≤1` solo es informativa a N bajo (5, 10); a N alto casi no quedan débiles que añadir.
2. **Superar el baseline de Italia** (Fibrina 0.333 / Granulación 0.786 / Callo 0.515) con
   FPN+MobileNetV2 + receta fuerte (aumentación fuerte + pérdida Tversky + sobre-muestreo de fibrina).
3. **Comparación de modelos** con la receta fuerte: FPN+MobileNetV2 (embebido) vs Unet++ / Unet+EfficientNet / SegFormer (cotas).

Todo el código vive en el repo público; este notebook solo lo orquesta.

---

## Cómo partir esto en sesiones de Colab

Una sesión de Colab gratis se corta por inactividad (~90 min sin interacción) y tiene un tope duro
(~12 h, en la práctica menos, y la GPU se puede retirar). El estudio completo son **~4–5 h en T4**, así
que conviene partirlo. El experimento 1 acumula todo en **la misma carpeta de Drive**
(`--dir_salida $EST`) y **reanuda** (salta lo ya hecho), de modo que se puede correr por bloques en
sesiones distintas sin perder nada:

| Bloque | Celdas | Tiempo aprox. T4 |
|---|---|---|
| **A** | Preparación + Exp. 1, **semilla 42** | ~75–90 min |
| **B** | Preparación + Exp. 1, **semilla 1** | ~75–90 min |
| **C** | Preparación + Exp. 1, **semilla 7** + figura de eficiencia | ~75–90 min |
| **D** | Preparación + Exp. 2 + Exp. 3 + tabla resumen | ~1.5–2 h |

En cada sesión nueva se re-ejecutan las celdas de **Preparación** (clonan el repo y re-enlazan Drive;
son idempotentes) y luego solo el bloque que toca. Si una sesión se corta a mitad, al re-lanzar la
misma celda continúa donde se quedó.

**Google AI Pro (Colab Pro):** sí sirve — permite ejecución en segundo plano y sesiones más largas
(L4/A100). Con Pro cabe todo en 1–2 sesiones y no hace falta partir por semilla; aun así los bloques
siguen funcionando igual.

## 1 · Preparación (clonar repo, instalar, bajar datos, montar Drive)

In [ ]:
import torch, os
print('CUDA disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecucion -> Cambiar tipo -> T4 GPU'

%cd /content
![ -d Co_MIL_PPS ] || git clone --depth 1 https://github.com/AdrianbeltranFC/Co_MIL_PPS
%pip -q install segmentation-models-pytorch==0.5.0

# Datos publicos (DFUTissue) -> se descargan frescos, no vienen en el repo
!cd Co_MIL_PPS && python CO-MIL/segmentacion/descargar_datos.py

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/comil_resultados_dfutissue'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Que todo lo que los scripts escriben en Pesos_Entrenados/ caiga directo en Drive
# (sobrevive a una desconexion de Colab)
!rm -rf Co_MIL_PPS/Pesos_Entrenados && ln -s {DRIVE_OUT} Co_MIL_PPS/Pesos_Entrenados

# Carpeta UNICA del estudio de eficiencia: los 3 bloques de semilla acumulan aqui y
# reanudan (saltan lo ya hecho). No cambiar el nombre entre sesiones.
EST = f'{DRIVE_OUT}/eficiencia_estudio'
os.makedirs(EST, exist_ok=True)
print('resultados ->', DRIVE_OUT)
print('estudio de eficiencia ->', EST)

In [ ]:
# Verificacion rapida del dataset
!cd Co_MIL_PPS && python CO-MIL/segmentacion/dataset_seg.py

## 2 · Experimento 1 — Eficiencia de anotación (¿aporta la débil? ¿conviene filtrarla?)

N &isin; {5, 10, 20, 40, 78} &times; 4 curvas (`solo_supervisado`, `mixto`, `mixto ≤1`, `mixto ≤2`)
&times; 5 semillas. Una celda **por semilla / grupo de semillas** para que quepa en una sesión de
Colab; todas escriben en `$EST` y **reanudan** (si una sesión se corta, re-lanza la misma celda).

- **Primera corrida (6-sep):** bloques A/B/C = semillas 42/1/7 → ✅ hecho, 54 corridas en `$EST`.
  Hallazgo: la anotación débil **no aporta** sobre las máscaras densas a esta escala, y filtrarla
  **empeora** (menos imágenes pesa más que la señal más limpia). La curva `solo_supervisado` es el
  resultado sólido: ~20 máscaras densas ≈ 78 % del máximo.
- **Bloque A2:** semillas 2 y 3, para pasar de 3 a 5 semillas (la varianza a N bajo es alta).

`--max_tejidos_debil none,1,2` genera las tres variantes `mixto` de una vez; el script salta solo
los puntos donde el filtro deja 0 débiles (p. ej. N=78). ~75–90 min por semilla en T4.

In [ ]:
# --- BLOQUE A: experimento 1, semilla 42 ---
!cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_eficiencia.py \
    --dir_salida {EST} \
    --mascaras 5,10,20,40,78 --semillas 42 --modos mixto,solo_supervisado \
    --max_tejidos_debil none,1,2 --epocas 150 --paciencia 25

In [ ]:
# --- BLOQUE B: experimento 1, semilla 1 ---
!cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_eficiencia.py \
    --dir_salida {EST} \
    --mascaras 5,10,20,40,78 --semillas 1 --modos mixto,solo_supervisado \
    --max_tejidos_debil none,1,2 --epocas 150 --paciencia 25

In [ ]:
# --- BLOQUE C: experimento 1, semilla 7 ---
!cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_eficiencia.py \
    --dir_salida {EST} \
    --mascaras 5,10,20,40,78 --semillas 7 --modos mixto,solo_supervisado \
    --max_tejidos_debil none,1,2 --epocas 150 --paciencia 25

In [ ]:
# --- BLOQUE A2: experimento 1, semillas extra (2 y 3) para llegar a 5 semillas ---
# Reanuda sobre $EST: solo entrena lo que falta (semillas 2 y 3); las 3 anteriores se saltan.
# ~2.5-3 h en T4. Después, la celda de resumen re-agrega las 5 semillas automáticamente.
!cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_eficiencia.py \
    --dir_salida {EST} \
    --mascaras 5,10,20,40,78 --semillas 2,3 --modos mixto,solo_supervisado \
    --max_tejidos_debil none,1,2 --epocas 150 --paciencia 25

## 3 · Experimento 2 — Superar el baseline de Italia (receta fuerte, 78 máscaras densas)

FPN + MobileNetV2, supervisión densa completa, con: aumentación fuerte + pérdida **Tversky**
(castiga más los falsos negativos, buena para la fibrina) + **sobre-muestreo** de las imágenes
con fibrina. 3 semillas para tener media &plusmn; desviación. ~30&ndash;45 min en T4.

In [ ]:
for s in (42, 1, 7):
    !cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_seg.py \
        --arch FPN --encoder mobilenet_v2 --perdida tversky --aug_fuerte --sobremuestreo \
        --epocas 300 --paciencia 45 --semilla {s}

## 4 · Experimento 3 — Comparación de modelos (misma receta fuerte)

El de MobileNetV2 (Exp. 2) es el **titular embebido**. Estos son cotas superiores:
Unet++ con ResNet34 (estilo Italia), Unet con EfficientNet-B0, y SegFormer-B0
(familia del paper original de DFUTissue). ~1 h en T4.

In [ ]:
modelos = [
    ('UnetPlusPlus', 'resnet34'),
    ('Unet', 'efficientnet-b0'),
    ('Segformer', 'mit_b0'),
]
for arch, enc in modelos:
    !cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_seg.py \
        --arch {arch} --encoder {enc} --perdida tversky --aug_fuerte --sobremuestreo \
        --epocas 300 --paciencia 45 --semilla 42

## 5 · Recolectar todo y hacer la figura resumen

**(Parte del Bloque D.)** Junta los tres bloques de semilla del Exp. 1 (curva de 4 líneas) y la tabla
de los Exp. 2/3 frente al baseline de Italia. Se puede correr aunque falten bloques: avisa de lo que
falta.

In [ ]:
import json, glob, sys, numpy as np, matplotlib.pyplot as plt
sys.path.append('/content/Co_MIL_PPS/CO-MIL/segmentacion')
from entrenar_eficiencia import agregar_por_n, _LBL_CURVA, _COL_CURVA

ITALIA = {'Fibrina': 0.333, 'Granulación': 0.786, 'Callo': 0.515}
ITALIA_MEDIA = float(np.mean(list(ITALIA.values())))

# ============ Experimentos 2 y 3: segmentación densa vs. Italia ============
filas = []
for md in sorted(glob.glob(f'{DRIVE_OUT}/seg_exp_*/metadata.json')):
    d = json.load(open(md))
    t = d['test']['dice_por_clase']
    filas.append((d['arquitectura'], d.get('semilla'), t['Fibrina'], t['Granulación'],
                  t['Callo'], d['test']['dice_medio_tejidos']))
if filas:
    print(f"{'modelo':<34}{'sem':>4}{'Fib':>8}{'Gra':>8}{'Cal':>8}{'Media':>8}")
    for f in filas:
        print(f"{f[0]:<34}{str(f[1]):>4}{f[2]:>8.3f}{f[3]:>8.3f}{f[4]:>8.3f}{f[5]:>8.3f}")
    print(f"{'ITALIA (ResUnet+Unet++)':<34}{'':>4}{ITALIA['Fibrina']:>8.3f}"
          f"{ITALIA['Granulación']:>8.3f}{ITALIA['Callo']:>8.3f}{ITALIA_MEDIA:>8.3f}")
else:
    print('(sin resultados de seg_exp_* todavía — corre el Bloque D)')

# ============ Experimento 1: curva de eficiencia (4 curvas) ============
# Junta TODOS los runs de la carpeta del estudio (los 3 bloques de semilla) y reagrega.
runs = []
for rj in sorted(set(glob.glob(f'{EST}/resultados.json') +
                     glob.glob(f'{DRIVE_OUT}/eficiencia_*/resultados.json'))):
    runs += json.load(open(rj)).get('runs', [])
# dedupe por (curva, N, semilla)
vistos, unicos = set(), []
for r in runs:
    k = (r.get('curva'), r['n_mascaras'], r.get('semilla'))
    if k not in vistos:
        vistos.add(k); unicos.append(r)
runs = unicos

if runs:
    ag = agregar_por_n(runs)
    orden = ['solo_supervisado', 'mixto', 'mixto_max2', 'mixto_max1']
    curvas = [c for c in orden if any(a['curva'] == c for a in ag)]
    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    for curva in curvas:
        sub = sorted([a for a in ag if a['curva'] == curva], key=lambda a: a['n_mascaras'])
        ns = np.array([a['n_mascaras'] for a in sub])
        m = np.array([a['dice_medio_tejidos_media'] for a in sub])
        s = np.array([a['dice_medio_tejidos_std'] for a in sub])
        ls = '--' if curva == 'solo_supervisado' else '-'
        ax.plot(ns, m, 'o', ls=ls, lw=2, color=_COL_CURVA.get(curva), label=_LBL_CURVA.get(curva, curva))
        ax.fill_between(ns, m - s, m + s, color=_COL_CURVA.get(curva), alpha=0.15)
    ax.axhline(ITALIA_MEDIA, color='#888', ls=':', label='baseline Italia (media)')
    n_sem = max(a['n_semillas'] for a in ag)
    ax.set_xlabel('Nº de imágenes con máscara densa (de 78)')
    ax.set_ylabel('Dice medio en tejidos (test)')
    ax.set_title(f'Eficiencia de anotación — DFUTissue ({n_sem} semillas, banda = ±1σ)')
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(f'{DRIVE_OUT}/resumen_eficiencia.png', dpi=140, bbox_inches='tight')
    plt.show()

    print(f"\n{'curva':>18} {'N':>3} {'nD':>4} | {'Fibrina':>13} {'Granul.':>13} {'Callo':>13} {'Media':>13}")
    for a in ag:
        mm = lambda c: f"{a['dice_'+c+'_media']:.3f}±{a['dice_'+c+'_std']:.3f}"
        print(f"{a['curva']:>18} {a['n_mascaras']:>3} {a['n_debil_medio']:>4.0f} | "
              f"{mm('Fibrina'):>13} {mm('Granulación'):>13} {mm('Callo'):>13} "
              f"{a['dice_medio_tejidos_media']:.3f}±{a['dice_medio_tejidos_std']:.3f}")
else:
    print('\n(sin resultados de eficiencia todavía — corre los Bloques A/B/C)')

print('\nTodo guardado en', DRIVE_OUT, '- descarga esa carpeta y avisa.')